# PointNeXt: Training on Kaggle

This notebook provides a complete workflow for training **PointNeXt** models on Kaggle.

**Supported tasks:**
- 3D Object Classification (ModelNet40, ScanObjectNN)
- Semantic Segmentation (S3DIS)
- Part Segmentation (ShapeNetPart)
- Custom Segmentation (NailSeg)

**Reference:** [PointNeXt: Revisiting PointNet++ with Improved Training and Scaling Strategies](https://arxiv.org/abs/2206.04670) (NeurIPS 2022)

> **Kaggle Tips:**
> - Enable **GPU** in *Settings → Accelerator → GPU T4 x2* (or P100).
> - Enable **Internet** in *Settings → Internet → On* for dependency installation.
> - Use Kaggle Datasets to upload large data (e.g., S3DIS, ScanObjectNN) to avoid re-downloading.

## 1. Environment Setup

Install all dependencies and build CUDA extensions required by PointNeXt.

### 1.1 Clone Repository & Initialize Submodules

In [ ]:
import os

# Clone the repository (skip if already cloned, e.g., via Kaggle Dataset)
REPO_DIR = '/kaggle/working/PointNeXt'
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/frank2033/PointNeXt.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

### 1.2 Install Python Dependencies

In [ ]:
import torch

# Install core dependencies
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__.split('+')[0]}+cu118.html
!pip install -q ninja easydict==1.9 pyyaml==6.0 h5py==3.6.0 scikit-learn==1.0.2 \
    tensorboard==2.8.0 tqdm wandb pyvista pandas Cython shortuuid multimethod

### 1.3 Build CUDA Extensions

Build the PointNet++ CUDA kernels. This is required for all tasks.

In [ ]:
# Build PointNet++ batch CUDA operations (required)
os.chdir(os.path.join(REPO_DIR, 'openpoints', 'cpp', 'pointnet2_batch'))
!python setup.py install --user 2>&1 | tail -5

# Build subsampling (needed for S3DIS voxel-based training)
os.chdir(os.path.join(REPO_DIR, 'openpoints', 'cpp', 'subsampling'))
!python setup.py build_ext --inplace 2>&1 | tail -5

# Build point transformer ops (optional, needed for Point Transformer models)
os.chdir(os.path.join(REPO_DIR, 'openpoints', 'cpp', 'pointops'))
!python setup.py install --user 2>&1 | tail -5

# Return to repo root
os.chdir(REPO_DIR)
print('CUDA extensions built successfully.')

### 1.4 Verify Installation

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

import sys
sys.path.insert(0, REPO_DIR)
from openpoints.utils import EasyConfig
print('\nOpenPoints library loaded successfully!')

## 2. Configuration System

PointNeXt uses a YAML-based hierarchical config system:
- `cfgs/default.yaml` — global defaults (optimizer, scheduler, logging)
- `cfgs/<task>/default.yaml` — task/dataset defaults
- `cfgs/<task>/<model>.yaml` — model-specific config (overrides task defaults)

Configs are loaded with `EasyConfig` and can be overridden programmatically.

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)

from openpoints.utils import EasyConfig

def load_config(cfg_path):
    """Load a PointNeXt config file with all defaults merged."""
    cfg = EasyConfig()
    cfg.load(cfg_path, recursive=True)
    return cfg

# Example: load ModelNet40 PointNeXt-S config
cfg = load_config(os.path.join(REPO_DIR, 'cfgs/modelnet40ply2048/pointnext-s.yaml'))
print('=== ModelNet40 PointNeXt-S Config ===')
print(f'Dataset: {cfg.dataset.common.NAME}')
print(f'Num classes: {cfg.num_classes}')
print(f'Model: {cfg.model.NAME}')
print(f'Encoder: {cfg.model.encoder_args.NAME}')
print(f'Epochs: {cfg.epochs}')
print(f'Batch size: {cfg.batch_size}')
print(f'Learning rate: {cfg.lr}')

## 3. Helper Functions

Common setup logic shared across all training tasks.

In [ ]:
import yaml
import logging
import numpy as np
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

from openpoints.utils import (
    EasyConfig, dist_utils, set_random_seed, save_checkpoint,
    load_checkpoint, resume_checkpoint, setup_logger_dist,
    cal_model_parm_nums, Wandb, generate_exp_directory,
    resume_exp_directory, AverageMeter, ConfusionMatrix, get_mious
)
from openpoints.dataset import build_dataloader_from_cfg, get_features_by_keys
from openpoints.transforms import build_transforms_from_cfg
from openpoints.optim import build_optimizer_from_cfg
from openpoints.scheduler import build_scheduler_from_cfg
from openpoints.loss import build_criterion_from_cfg
from openpoints.models import build_model_from_cfg


def setup_cfg_for_notebook(cfg_path, overrides=None):
    """
    Load and configure a PointNeXt config for single-GPU Kaggle training.

    Args:
        cfg_path: Path to the YAML config file.
        overrides: Dict of config overrides, e.g. {'epochs': 50, 'batch_size': 16}.

    Returns:
        Configured EasyConfig object ready for training.
    """
    cfg = EasyConfig()
    cfg.load(cfg_path, recursive=True)

    # Apply overrides
    if overrides:
        for k, v in overrides.items():
            cfg[k] = v

    if cfg.seed is None:
        cfg.seed = np.random.randint(1, 10000)

    # Single GPU setup (no distributed)
    cfg.rank = 0
    cfg.world_size = 1
    cfg.distributed = False
    cfg.mp = False
    cfg.sync_bn = False

    # Disable wandb by default on Kaggle (set True if you have an API key)
    cfg.wandb.use_wandb = False

    # Derive task name and experiment name from cfg_path
    cfg.task_name = cfg_path.split('.')[-2].split('/')[-2]
    cfg.cfg_basename = cfg_path.split('.')[-2].split('/')[-1]
    cfg.exp_name = cfg.cfg_basename
    tags = [cfg.task_name, cfg.mode, cfg.cfg_basename,
            f'ngpus{cfg.world_size}', f'seed{cfg.seed}']
    cfg.opts = ''

    # Set root dir and generate experiment directory
    cfg.root_dir = os.path.join(cfg.root_dir, cfg.task_name)
    cfg.is_training = cfg.mode not in ['test', 'testing', 'val', 'eval', 'evaluation']
    generate_exp_directory(cfg, tags, additional_id=None)

    os.environ['JOB_LOG_DIR'] = cfg.log_dir
    cfg.cfg_path = os.path.join(cfg.run_dir, 'cfg.yaml')
    with open(cfg.cfg_path, 'w') as f:
        yaml.dump(cfg, f, indent=2)

    return cfg


def plot_training_curves(history, title='Training Curves'):
    """Plot training loss and accuracy/mIoU curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = history['epochs']
    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[0].grid(True)

    for key in history:
        if key not in ('epochs', 'train_loss', 'lr') and len(history[key]) == len(epochs):
            axes[1].plot(epochs, history[key], label=key)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric')
    axes[1].set_title(f'{title} - Metrics')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


print('Helper functions defined.')

## 4. Task A: 3D Object Classification (ModelNet40)

ModelNet40 contains 12,311 meshed 3D CAD models from 40 categories.
The dataset is **auto-downloaded** on first use.

- **Input**: 1024 points (xyz)
- **Model**: PointNeXt-S (~1.4M params)
- **Metric**: Overall Accuracy (OA)

### 4.1 Configure & Load Data

In [ ]:
os.chdir(REPO_DIR)

# Load config — reduce epochs for quick demo; increase for full training
cls_cfg_path = os.path.join(REPO_DIR, 'cfgs/modelnet40ply2048/pointnext-s.yaml')
cls_cfg = setup_cfg_for_notebook(cls_cfg_path, overrides={
    'epochs': 100,         # Full training: 600
    'batch_size': 32,
    'val_freq': 5,
    'dataloader': {'num_workers': 2},  # Kaggle has limited CPU cores
})

setup_logger_dist(cls_cfg.log_path, cls_cfg.rank, name=cls_cfg.dataset.common.NAME)
set_random_seed(cls_cfg.seed, deterministic=cls_cfg.deterministic)
torch.backends.cudnn.enabled = True

print(f'Config loaded: {cls_cfg_path}')
print(f'Log dir: {cls_cfg.run_dir}')

### 4.2 Build Model, Optimizer & Data Loaders

In [ ]:
from openpoints.models.layers import furthest_point_sample

# Build model
if not cls_cfg.model.get('criterion_args', False):
    cls_cfg.model.criterion_args = cls_cfg.criterion_args
cls_model = build_model_from_cfg(cls_cfg.model).cuda()
model_size = cal_model_parm_nums(cls_model)
print(f'Model: {cls_cfg.model.NAME}, Params: {model_size / 1e6:.4f} M')

if cls_cfg.model.get('in_channels', None) is None:
    cls_cfg.model.in_channels = cls_cfg.model.encoder_args.in_channels

# Build optimizer & scheduler
cls_optimizer = build_optimizer_from_cfg(cls_model, lr=cls_cfg.lr, **cls_cfg.optimizer)
cls_scheduler = build_scheduler_from_cfg(cls_cfg, cls_optimizer)

# Build data loaders
cls_train_loader = build_dataloader_from_cfg(
    cls_cfg.batch_size, cls_cfg.dataset, cls_cfg.dataloader,
    datatransforms_cfg=cls_cfg.datatransforms, split='train', distributed=False)
cls_val_loader = build_dataloader_from_cfg(
    cls_cfg.get('val_batch_size', cls_cfg.batch_size), cls_cfg.dataset, cls_cfg.dataloader,
    datatransforms_cfg=cls_cfg.datatransforms, split='val', distributed=False)

num_classes = cls_val_loader.dataset.num_classes if hasattr(cls_val_loader.dataset, 'num_classes') else None
cls_cfg.classes = cls_val_loader.dataset.classes if hasattr(cls_val_loader.dataset, 'classes') else list(range(num_classes))

print(f'Train samples: {len(cls_train_loader.dataset)}')
print(f'Val samples: {len(cls_val_loader.dataset)}')
print(f'Num classes: {num_classes}')

### 4.3 Train Classification Model

In [ ]:
def cls_train_one_epoch(model, train_loader, optimizer, scheduler, epoch, cfg):
    """Train one epoch for classification."""
    loss_meter = AverageMeter()
    cm = ConfusionMatrix(num_classes=cfg.num_classes)
    npoints = cfg.num_points
    model.train()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train E{epoch}')
    num_iter = 0
    for idx, data in pbar:
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        num_iter += 1
        points = data['x']
        target = data['y']
        num_curr_pts = points.shape[1]
        if num_curr_pts > npoints:
            if npoints == 1024:
                point_all = 1200
            elif npoints == 4096:
                point_all = 4800
            elif npoints == 8192:
                point_all = 8192
            else:
                point_all = int(npoints * 1.2)
            if points.size(1) < point_all:
                point_all = points.size(1)
            fps_idx = furthest_point_sample(points[:, :, :3].contiguous(), point_all)
            fps_idx = fps_idx[:, np.random.choice(point_all, npoints, False)]
            points = torch.gather(points, 1, fps_idx.unsqueeze(-1).long().expand(-1, -1, points.shape[-1]))
        data['pos'] = points[:, :, :3].contiguous()
        data['x'] = points[:, :, :cfg.model.in_channels].transpose(1, 2).contiguous()
        logits, loss = model.get_logits_loss(data, target) if not hasattr(model, 'module') else model.module.get_logits_loss(data, target)
        loss.backward()
        if num_iter == cfg.step_per_update:
            if cfg.get('grad_norm_clip') is not None and cfg.grad_norm_clip > 0.:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_norm_clip, norm_type=2)
            num_iter = 0
            optimizer.step()
            model.zero_grad()
            if not cfg.sched_on_epoch:
                scheduler.step(epoch)
        cm.update(logits.argmax(dim=1), target)
        loss_meter.update(loss.item())
        if idx % cfg.print_freq == 0:
            pbar.set_description(f'Train E[{epoch}/{cfg.epochs}] Loss {loss_meter.val:.3f} Acc {cm.overall_accuray:.2f}')
    macc, oa, accs = cm.all_acc()
    return loss_meter.avg, macc, oa, accs


@torch.no_grad()
def cls_validate(model, val_loader, cfg):
    """Validate classification model."""
    model.eval()
    cm = ConfusionMatrix(num_classes=cfg.num_classes)
    npoints = cfg.num_points
    for idx, data in tqdm(enumerate(val_loader), total=len(val_loader), desc='Val'):
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        target = data['y']
        points = data['x'][:, :npoints]
        data['pos'] = points[:, :, :3].contiguous()
        data['x'] = points[:, :, :cfg.model.in_channels].transpose(1, 2).contiguous()
        logits = model(data)
        cm.update(logits.argmax(dim=1), target)
    macc, oa, accs = cm.all_acc()
    return macc, oa, accs


print('Classification train/val functions defined.')

In [ ]:
# ---- Training Loop ----
cls_history = {'epochs': [], 'train_loss': [], 'train_oa': [], 'val_oa': []}
best_val_oa, best_epoch = 0., 0
cls_model.zero_grad()

for epoch in range(cls_cfg.start_epoch, cls_cfg.epochs + 1):
    if hasattr(cls_train_loader.dataset, 'epoch'):
        cls_train_loader.dataset.epoch = epoch - 1
    train_loss, train_macc, train_oa, _ = cls_train_one_epoch(
        cls_model, cls_train_loader, cls_optimizer, cls_scheduler, epoch, cls_cfg)

    val_oa = 0.
    if epoch % cls_cfg.val_freq == 0:
        val_macc, val_oa, val_accs = cls_validate(cls_model, cls_val_loader, cls_cfg)
        is_best = val_oa > best_val_oa
        if is_best:
            best_val_oa = val_oa
            best_epoch = epoch
            print(f'\n*** New best @E{epoch}: OA={val_oa:.2f}, mAcc={val_macc:.2f} ***')
        save_checkpoint(cls_cfg, cls_model, epoch, cls_optimizer, cls_scheduler,
                        additioanl_dict={'best_val': best_val_oa}, is_best=is_best)

    if cls_cfg.sched_on_epoch:
        cls_scheduler.step(epoch)

    lr = cls_optimizer.param_groups[0]['lr']
    cls_history['epochs'].append(epoch)
    cls_history['train_loss'].append(train_loss)
    cls_history['train_oa'].append(float(train_oa))
    cls_history['val_oa'].append(float(val_oa))

    if epoch % 10 == 0:
        print(f'E{epoch} LR={lr:.6f} train_oa={train_oa:.2f} val_oa={val_oa:.2f} best={best_val_oa:.2f}')

print(f'\nTraining complete! Best val OA: {best_val_oa:.2f} @epoch {best_epoch}')

### 4.4 Visualize Classification Results

In [ ]:
plot_training_curves(cls_history, title='ModelNet40 Classification')

## 5. Task B: Semantic Segmentation (S3DIS)

S3DIS (Stanford 3D Indoor Scenes) contains 3D scans of 6 indoor areas with 13 semantic classes.

- **Input**: Points with XYZ + RGB + height features
- **Model**: PointNeXt-S (~0.8M params)
- **Metric**: mIoU, OA, mAcc

> **Data Preparation**: S3DIS requires pre-processing. Follow the instructions in
> `docs/examples/s3dis.md` or download the preprocessed data and place it in `data/S3DIS/s3disfull/`.
> On Kaggle, upload the preprocessed data as a Kaggle Dataset.

In [ ]:
os.chdir(REPO_DIR)

# If your S3DIS data is in a Kaggle Dataset, create a symlink:
# !ln -s /kaggle/input/s3dis-preprocessed data/S3DIS

seg_cfg_path = os.path.join(REPO_DIR, 'cfgs/s3dis/pointnext-s.yaml')
seg_cfg = setup_cfg_for_notebook(seg_cfg_path, overrides={
    'epochs': 50,           # Full training: 100
    'batch_size': 16,       # Reduce if OOM on Kaggle
    'val_freq': 5,
    'dataloader': {'num_workers': 2},
})

setup_logger_dist(seg_cfg.log_path, seg_cfg.rank, name=seg_cfg.dataset.common.NAME)
set_random_seed(seg_cfg.seed, deterministic=seg_cfg.deterministic)

print(f'Config loaded: {seg_cfg_path}')
print(f'Dataset: {seg_cfg.dataset.common.NAME}')
print(f'Num classes: {seg_cfg.num_classes}')

In [ ]:
# Build model
if seg_cfg.model.get('in_channels', None) is None:
    seg_cfg.model.in_channels = seg_cfg.model.encoder_args.in_channels
seg_model = build_model_from_cfg(seg_cfg.model).cuda()
print(f'Seg model params: {cal_model_parm_nums(seg_model) / 1e6:.4f} M')

seg_optimizer = build_optimizer_from_cfg(seg_model, lr=seg_cfg.lr, **seg_cfg.optimizer)
seg_scheduler = build_scheduler_from_cfg(seg_cfg, seg_optimizer)

# Build data loaders (requires S3DIS data to be present)
try:
    seg_train_loader = build_dataloader_from_cfg(
        seg_cfg.batch_size, seg_cfg.dataset, seg_cfg.dataloader,
        datatransforms_cfg=seg_cfg.datatransforms, split='train', distributed=False)
    seg_val_loader = build_dataloader_from_cfg(
        seg_cfg.get('val_batch_size', seg_cfg.batch_size), seg_cfg.dataset, seg_cfg.dataloader,
        datatransforms_cfg=seg_cfg.datatransforms, split='val', distributed=False)
    print(f'Train samples: {len(seg_train_loader.dataset)}')
    print(f'Val samples: {len(seg_val_loader.dataset)}')
    seg_data_ready = True
except Exception as e:
    print(f'S3DIS data not found: {e}')
    print('Please download and preprocess S3DIS data first. See docs/examples/s3dis.md')
    seg_data_ready = False

### 5.1 Train Segmentation Model

In [ ]:
def seg_train_one_epoch(model, train_loader, criterion, optimizer, scheduler, epoch, cfg):
    """Train one epoch for segmentation."""
    loss_meter = AverageMeter()
    cm = ConfusionMatrix(num_classes=cfg.num_classes, ignore_index=cfg.ignore_index)
    model.train()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train E{epoch}')
    num_iter = 0
    for idx, data in pbar:
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        num_iter += 1
        target = data['y']
        data['x'] = get_features_by_keys(data, cfg.feature_keys)
        logits = model(data)
        loss = criterion(logits, target)
        loss.backward()
        if num_iter == cfg.step_per_update:
            if cfg.get('grad_norm_clip') is not None and cfg.grad_norm_clip > 0.:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_norm_clip, norm_type=2)
            num_iter = 0
            optimizer.step()
            optimizer.zero_grad()
            if not cfg.sched_on_epoch:
                scheduler.step(epoch)
        cm.update(logits.argmax(dim=1), target)
        loss_meter.update(loss.item())
        if idx % cfg.print_freq == 0:
            pbar.set_description(f'Train E[{epoch}/{cfg.epochs}] Loss {loss_meter.val:.3f} Acc {cm.overall_accuray:.2f}')
    return loss_meter.avg


@torch.no_grad()
def seg_validate(model, val_loader, cfg):
    """Validate segmentation model."""
    model.eval()
    cm = ConfusionMatrix(num_classes=cfg.num_classes, ignore_index=cfg.ignore_index)
    for idx, data in tqdm(enumerate(val_loader), total=len(val_loader), desc='Val'):
        for key in data.keys():
            data[key] = data[key].cuda(non_blocking=True)
        target = data['y']
        data['x'] = get_features_by_keys(data, cfg.feature_keys)
        logits = model(data)
        cm.update(logits.argmax(dim=1), target)
    tp, union, count = cm.tp, cm.union, cm.count
    miou, macc, oa, ious, accs = get_mious(tp, union, count)
    return miou, macc, oa, ious, accs


print('Segmentation train/val functions defined.')

In [ ]:
# ---- Segmentation Training Loop ----
if seg_data_ready:
    from openpoints.dataset import get_class_weights

    if seg_cfg.get('cls_weighed_loss', False) and hasattr(seg_train_loader.dataset, 'num_per_class'):
        seg_cfg.criterion_args.weight = get_class_weights(seg_train_loader.dataset.num_per_class, normalize=True)
    seg_criterion = build_criterion_from_cfg(seg_cfg.criterion_args).cuda()

    seg_history = {'epochs': [], 'train_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou, best_epoch = 0., 0

    for epoch in range(seg_cfg.start_epoch, seg_cfg.epochs + 1):
        if hasattr(seg_train_loader.dataset, 'epoch'):
            seg_train_loader.dataset.epoch = epoch - 1
        seg_cfg.epoch = epoch

        train_loss = seg_train_one_epoch(
            seg_model, seg_train_loader, seg_criterion,
            seg_optimizer, seg_scheduler, epoch, seg_cfg)

        val_miou, val_oa = 0., 0.
        if epoch % seg_cfg.val_freq == 0:
            val_miou, val_macc, val_oa, val_ious, _ = seg_validate(seg_model, seg_val_loader, seg_cfg)
            is_best = val_miou > best_miou
            if is_best:
                best_miou = val_miou
                best_epoch = epoch
                print(f'\n*** New best @E{epoch}: mIoU={val_miou:.2f}, OA={val_oa:.2f} ***')
            save_checkpoint(seg_cfg, seg_model, epoch, seg_optimizer, seg_scheduler,
                            additioanl_dict={'best_val': best_miou}, is_best=is_best)

        if seg_cfg.sched_on_epoch:
            seg_scheduler.step(epoch)

        seg_history['epochs'].append(epoch)
        seg_history['train_loss'].append(train_loss)
        seg_history['val_miou'].append(float(val_miou))
        seg_history['val_oa'].append(float(val_oa))

    print(f'\nSeg training complete! Best mIoU: {best_miou:.2f} @epoch {best_epoch}')
    plot_training_curves(seg_history, title='S3DIS Segmentation')
else:
    print('Skipping S3DIS training — data not available.')

## 6. Task C: Part Segmentation (ShapeNetPart)

ShapeNetPart contains 16,881 shapes from 16 categories with 50 part labels.

- **Input**: 2048 points with XYZ + normals
- **Model**: PointNeXt-S (BasePartSeg, ~1.0M params)
- **Metric**: Instance mIoU, Class mIoU

> Place ShapeNetPart data in `data/ShapeNetPart/shapenetcore_partanno_segmentation_benchmark_v0_normal/`
> or create a symlink from your Kaggle Dataset.

In [ ]:
os.chdir(REPO_DIR)

# Symlink Kaggle Dataset if needed:
# !mkdir -p data/ShapeNetPart
# !ln -s /kaggle/input/shapenet-part data/ShapeNetPart/shapenetcore_partanno_segmentation_benchmark_v0_normal

part_cfg_path = os.path.join(REPO_DIR, 'cfgs/shapenetpart/pointnext-s.yaml')
part_cfg = setup_cfg_for_notebook(part_cfg_path, overrides={
    'epochs': 100,       # Full training: 300
    'batch_size': 8,
    'val_freq': 10,
    'dataloader': {'num_workers': 2},
})

setup_logger_dist(part_cfg.log_path, part_cfg.rank, name=part_cfg.dataset.common.NAME)
set_random_seed(part_cfg.seed, deterministic=part_cfg.deterministic)

print(f'Config loaded: {part_cfg_path}')
print(f'Num part classes: {part_cfg.num_classes}')
print(f'Num shape classes: {part_cfg.shape_classes}')

In [ ]:
# Build data loaders (requires ShapeNetPart data)
try:
    part_val_loader = build_dataloader_from_cfg(
        part_cfg.batch_size, part_cfg.dataset, part_cfg.dataloader,
        datatransforms_cfg=part_cfg.datatransforms, split='val', distributed=False)
    part_cfg.cls2parts = part_val_loader.dataset.cls2parts
    if part_cfg.model.get('decoder_args', False):
        part_cfg.model.decoder_args.cls2partembed = part_val_loader.dataset.cls2partembed
    if part_cfg.model.get('in_channels', None) is None:
        part_cfg.model.in_channels = part_cfg.model.encoder_args.in_channels

    part_model = build_model_from_cfg(part_cfg.model).cuda()
    print(f'Part seg model params: {cal_model_parm_nums(part_model) / 1e6:.4f} M')

    part_train_loader = build_dataloader_from_cfg(
        part_cfg.batch_size, part_cfg.dataset, part_cfg.dataloader,
        datatransforms_cfg=part_cfg.datatransforms, split='train', distributed=False)
    print(f'Train samples: {len(part_train_loader.dataset)}')
    print(f'Val samples: {len(part_val_loader.dataset)}')
    part_data_ready = True
except Exception as e:
    print(f'ShapeNetPart data not found: {e}')
    print('Please download ShapeNetPart data. See docs/examples/shapenetpart.md')
    part_data_ready = False

### 6.1 Train Part Segmentation Model

The training loop for ShapeNetPart part segmentation. Uses the same
segmentation training functions with the part segmentation criterion.

In [ ]:
if part_data_ready:
    part_optimizer = build_optimizer_from_cfg(part_model, lr=part_cfg.lr, **part_cfg.optimizer)
    part_scheduler = build_scheduler_from_cfg(part_cfg, part_optimizer)
    part_criterion = build_criterion_from_cfg(part_cfg.criterion_args).cuda()

    part_history = {'epochs': [], 'train_loss': []}
    best_ins_miou, best_epoch = 0., 0

    for epoch in range(part_cfg.start_epoch, part_cfg.epochs + 1):
        if hasattr(part_train_loader.dataset, 'epoch'):
            part_train_loader.dataset.epoch = epoch - 1
        part_cfg.epoch = epoch

        # Train
        train_loss = seg_train_one_epoch(
            part_model, part_train_loader, part_criterion,
            part_optimizer, part_scheduler, epoch, part_cfg)

        if part_cfg.sched_on_epoch:
            part_scheduler.step(epoch)

        part_history['epochs'].append(epoch)
        part_history['train_loss'].append(train_loss)

        if epoch % part_cfg.val_freq == 0:
            save_checkpoint(part_cfg, part_model, epoch, part_optimizer, part_scheduler,
                            additioanl_dict={'ins_miou': best_ins_miou}, is_best=False)
            print(f'E{epoch} train_loss={train_loss:.4f}')

    print(f'\nPart seg training complete!')
else:
    print('Skipping ShapeNetPart training — data not available.')

## 7. Task D: Custom Segmentation (NailSeg)

NailSeg is a custom fingernail point cloud segmentation task included in this repository.

- **Input**: Points with XYZ + height features
- **Model**: PointNeXt-S (BaseSeg, ~0.8M params)
- **Classes**: 2 (non-nail, nail)
- **Metric**: mIoU, OA, mAcc

> Place NailSeg data in `data/NailSeg/{train,val,test}/` with `.npy` files.
> Each file should be (N, C) where first 3 cols are XYZ and last col is label.

In [ ]:
os.chdir(REPO_DIR)

# Symlink Kaggle Dataset if needed:
# !ln -s /kaggle/input/nailseg-data data/NailSeg

# Register the NailSeg dataset class
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'examples', 'nailseg'))
from nailseg_dataset import NailSeg

nail_cfg_path = os.path.join(REPO_DIR, 'cfgs/nailseg/pointnext-s.yaml')
nail_cfg = setup_cfg_for_notebook(nail_cfg_path, overrides={
    'epochs': 100,       # Full training: 200
    'batch_size': 16,
    'val_freq': 5,
    'dataloader': {'num_workers': 2},
})

setup_logger_dist(nail_cfg.log_path, nail_cfg.rank, name=nail_cfg.dataset.common.NAME)
set_random_seed(nail_cfg.seed, deterministic=nail_cfg.deterministic)

print(f'Config loaded: {nail_cfg_path}')
print(f'Num classes: {nail_cfg.num_classes}')

In [ ]:
# Build model and data loaders for NailSeg
try:
    if nail_cfg.model.get('in_channels', None) is None:
        nail_cfg.model.in_channels = nail_cfg.model.encoder_args.in_channels
    nail_model = build_model_from_cfg(nail_cfg.model).cuda()
    print(f'NailSeg model params: {cal_model_parm_nums(nail_model) / 1e6:.4f} M')

    nail_train_loader = build_dataloader_from_cfg(
        nail_cfg.batch_size, nail_cfg.dataset, nail_cfg.dataloader,
        datatransforms_cfg=nail_cfg.datatransforms, split='train', distributed=False)
    nail_val_loader = build_dataloader_from_cfg(
        nail_cfg.get('val_batch_size', nail_cfg.batch_size), nail_cfg.dataset, nail_cfg.dataloader,
        datatransforms_cfg=nail_cfg.datatransforms, split='val', distributed=False)
    nail_cfg.classes = nail_val_loader.dataset.classes if hasattr(
        nail_val_loader.dataset, 'classes') else [str(i) for i in range(nail_cfg.num_classes)]
    print(f'Train samples: {len(nail_train_loader.dataset)}')
    print(f'Val samples: {len(nail_val_loader.dataset)}')
    nail_data_ready = True
except Exception as e:
    print(f'NailSeg data not found: {e}')
    print('Place .npy files in data/NailSeg/{train,val,test}/')
    nail_data_ready = False

In [ ]:
# ---- NailSeg Training Loop ----
if nail_data_ready:
    nail_optimizer = build_optimizer_from_cfg(nail_model, lr=nail_cfg.lr, **nail_cfg.optimizer)
    nail_scheduler = build_scheduler_from_cfg(nail_cfg, nail_optimizer)
    nail_criterion = build_criterion_from_cfg(nail_cfg.criterion_args).cuda()

    nail_history = {'epochs': [], 'train_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou, best_epoch = 0., 0

    for epoch in range(nail_cfg.start_epoch, nail_cfg.epochs + 1):
        if hasattr(nail_train_loader.dataset, 'epoch'):
            nail_train_loader.dataset.epoch = epoch - 1
        nail_cfg.epoch = epoch

        train_loss = seg_train_one_epoch(
            nail_model, nail_train_loader, nail_criterion,
            nail_optimizer, nail_scheduler, epoch, nail_cfg)

        val_miou, val_oa = 0., 0.
        if epoch % nail_cfg.val_freq == 0:
            val_miou, val_macc, val_oa, val_ious, _ = seg_validate(
                nail_model, nail_val_loader, nail_cfg)
            is_best = val_miou > best_miou
            if is_best:
                best_miou = val_miou
                best_epoch = epoch
                print(f'\n*** New best @E{epoch}: mIoU={val_miou:.2f}, OA={val_oa:.2f} ***')
            save_checkpoint(nail_cfg, nail_model, epoch, nail_optimizer, nail_scheduler,
                            additioanl_dict={'best_val': best_miou}, is_best=is_best)

        if nail_cfg.sched_on_epoch:
            nail_scheduler.step(epoch)

        nail_history['epochs'].append(epoch)
        nail_history['train_loss'].append(train_loss)
        nail_history['val_miou'].append(float(val_miou))
        nail_history['val_oa'].append(float(val_oa))

    print(f'\nNailSeg training complete! Best mIoU: {best_miou:.2f} @epoch {best_epoch}')
    plot_training_curves(nail_history, title='NailSeg Segmentation')
else:
    print('Skipping NailSeg training — data not available.')

## 8. Save Results & Download Checkpoints

Save the best checkpoints and training curves for download from Kaggle.

In [ ]:
import shutil

output_dir = '/kaggle/working/results'
os.makedirs(output_dir, exist_ok=True)

# Copy best checkpoints to output
log_root = os.path.join(REPO_DIR, 'log')
if os.path.exists(log_root):
    for root, dirs, files in os.walk(log_root):
        for f in files:
            if 'best' in f and f.endswith('.pth'):
                src = os.path.join(root, f)
                dst = os.path.join(output_dir, f)
                shutil.copy2(src, dst)
                print(f'Saved: {dst}')

print(f'\nAll results saved to {output_dir}')
print('Download from the Kaggle Output tab after the notebook finishes.')

## 9. Tips for Kaggle Training

### Memory Management
- Kaggle T4 GPU has **15 GB** VRAM. Reduce `batch_size` if you get OOM errors.
- For S3DIS, reduce `voxel_max` in the config (e.g., `dataset.train.voxel_max: 12000`).

### Training Time
- Kaggle sessions are limited to **12 hours** (GPU). Plan epochs accordingly.
- Use `mode=resume` with `pretrained_path` to continue training across sessions.

### Data Management
- Upload preprocessed datasets as **Kaggle Datasets** to avoid re-downloading.
- Use symlinks: `!ln -s /kaggle/input/<dataset-name> data/<DatasetFolder>`

### Saving Checkpoints
- Checkpoints are auto-saved to `log/<task>/`. Copy best ones to `/kaggle/working/`.
- Use `Save & Run All` → download from Output tab.
- For long training, save intermediate checkpoints to Kaggle Datasets.

### Config Overrides
- Override any config value via the `overrides` dict in `setup_cfg_for_notebook()`.
- Common overrides: `epochs`, `batch_size`, `lr`, `num_points`, `dataloader.num_workers`.

### Resume Training
```python
cfg = setup_cfg_for_notebook(cfg_path, overrides={'mode': 'resume'})
cfg.pretrained_path = '/kaggle/input/my-checkpoints/ckpt_best.pth'
resume_checkpoint(cfg, model, optimizer, scheduler, pretrained_path=cfg.pretrained_path)
```